# 🎬 Movie Recommendation System using Clustering

**Dataset:** TMDB 5000 Movies (Kaggle) · **Algorithm:** K-Means clustering

This notebook walks through the full pipeline:
1. Load & merge the data
2. Exploratory data analysis (EDA)
3. Feature engineering (genres, keywords, cast, director → TF-IDF + numeric)
4. Choosing the number of clusters (elbow + silhouette)
5. K-Means clustering & 2-D visualisation
6. Generating recommendations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from recommender import (
    load_data, engineer_features, build_feature_matrix,
    choose_k, fit_clusters, MovieRecommender,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)

## 1. Load & merge the data

In [ ]:
df = load_data()
print(f'{len(df):,} movies, {df.shape[1]} columns')
df[['title', 'genres', 'vote_average', 'vote_count', 'popularity']].head()

## 2. Exploratory data analysis

In [ ]:
df = engineer_features(df)

# Most common genres
genre_counts = df['genres_list'].explode().value_counts().head(12)
genre_counts.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title('Most common genres in TMDB 5000')
plt.xlabel('Number of movies')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df['vote_average'].plot(kind='hist', bins=30, ax=ax[0], color='coral')
ax[0].set_title('Distribution of ratings')
ax[0].set_xlabel('vote_average')
df['release_year'].dropna().plot(kind='hist', bins=40, ax=ax[1], color='seagreen')
ax[1].set_title('Movies per release year')
ax[1].set_xlabel('year')
plt.show()

## 3. Feature engineering

Each movie is turned into a text **"soup"** (genres ×3, keywords, top-3 cast,
director ×2, overview), vectorised with **TF-IDF** and compressed with
**Truncated SVD**. Numeric signals (rating, votes, popularity, runtime, year)
are log-scaled + min-max normalised and appended.

In [ ]:
print('Example soup:\n', df['soup'].iloc[0][:300], '...')

features, pieces = build_feature_matrix(df)
print('\nFeature matrix shape:', features.shape)

## 4. Choosing the number of clusters

We sweep *k* and look at the **elbow** (inertia) and the **silhouette score**.

In [ ]:
scores = choose_k(features, k_range=range(4, 16))
scores

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(scores['k'], scores['inertia'], 'o-', color='steelblue')
ax[0].set_title('Elbow method'); ax[0].set_xlabel('k'); ax[0].set_ylabel('inertia')
ax[1].plot(scores['k'], scores['silhouette'], 'o-', color='darkorange')
ax[1].set_title('Silhouette score'); ax[1].set_xlabel('k'); ax[1].set_ylabel('score')
plt.show()

best_k = int(scores.sort_values('silhouette', ascending=False).iloc[0]['k'])
print('Best k by silhouette:', best_k)

## 5. K-Means clustering & visualisation

In [ ]:
K = 10  # chosen for interpretable, well-sized genre themes
kmeans, labels = fit_clusters(features, n_clusters=K)
rec = MovieRecommender(df, features, labels, kmeans, pieces)

rec.cluster_summary()

In [ ]:
# 2-D projection of the clusters (Truncated SVD to 2 components)
from sklearn.decomposition import TruncatedSVD

coords = TruncatedSVD(n_components=2, random_state=42).fit_transform(features)
plt.figure(figsize=(9, 7))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=labels,
                      cmap='tab10', s=8, alpha=0.6)
plt.title('Movie clusters (2-D SVD projection)')
plt.xlabel('component 1'); plt.ylabel('component 2')
plt.colorbar(scatter, label='cluster')
plt.show()

## 6. Recommendations

In [ ]:
rec.recommend('The Dark Knight', n=10)

In [ ]:
rec.recommend('Toy Story', n=10)

In [ ]:
rec.recommend('The Avengers', n=10)

---
**Summary:** K-Means over a TF-IDF + numeric feature space groups ~4,800 movies
into coherent genre-driven clusters; recommendations are the nearest neighbours
*within* a movie's cluster. Run `streamlit run app.py` for the interactive demo.